In [90]:
import mne 
from nilearn import datasets
import nibabel
import numpy as np
from pathlib import Path

In [91]:
subjects_dir = mne.datasets.fetch_fsaverage(verbose=True)
mne.datasets.fetch_aparc_sub_parcellation(subjects_dir=subjects_dir)

0 files missing from root.txt in C:\Users\hugop\mne_data\MNE-fsaverage-data
0 files missing from bem.txt in C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage


In [92]:
subjects_dir = Path(subjects_dir).parent
subjects_dir

WindowsPath('C:/Users/hugop/mne_data/MNE-fsaverage-data')

In [93]:
subject='fsaverage'

In [94]:
labels = mne.read_labels_from_annot(
    subject=subject,
    parc='Schaefer2018_200Parcels_7Networks_order',
    subjects_dir=subjects_dir
)
labels

Reading labels from parcellation...
   read 101 labels from C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\label\lh.Schaefer2018_200Parcels_7Networks_order.annot
   read 101 labels from C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\label\rh.Schaefer2018_200Parcels_7Networks_order.annot


[<Label | fsaverage, '7Networks_LH_Cont_Cing_1-lh', lh : 1295 vertices>,
 <Label | fsaverage, '7Networks_LH_Cont_Cing_2-lh', lh : 714 vertices>,
 <Label | fsaverage, '7Networks_LH_Cont_OFC_1-lh', lh : 872 vertices>,
 <Label | fsaverage, '7Networks_LH_Cont_PFCl_1-lh', lh : 1089 vertices>,
 <Label | fsaverage, '7Networks_LH_Cont_PFCl_2-lh', lh : 1197 vertices>,
 <Label | fsaverage, '7Networks_LH_Cont_PFCl_3-lh', lh : 1668 vertices>,
 <Label | fsaverage, '7Networks_LH_Cont_PFCl_4-lh', lh : 1660 vertices>,
 <Label | fsaverage, '7Networks_LH_Cont_PFCl_5-lh', lh : 1255 vertices>,
 <Label | fsaverage, '7Networks_LH_Cont_Par_1-lh', lh : 1124 vertices>,
 <Label | fsaverage, '7Networks_LH_Cont_Par_2-lh', lh : 1696 vertices>,
 <Label | fsaverage, '7Networks_LH_Cont_Par_3-lh', lh : 1117 vertices>,
 <Label | fsaverage, '7Networks_LH_Cont_Temp_1-lh', lh : 972 vertices>,
 <Label | fsaverage, '7Networks_LH_Cont_pCun_1-lh', lh : 1226 vertices>,
 <Label | fsaverage, '7Networks_LH_Default_PFC_1-lh', lh :

In [95]:
def parse_label(label):
    name = label.name.replace('-lh', '').replace('-rh', '')
    parts = name.split('_')

    return {
        "network": parts[2],
    }

In [96]:
regions = {}

for lab in labels:
    info = parse_label(lab)
    reg = info["network"]
    
    if reg is None:
        continue
        
    regions.setdefault(reg, []).append(lab)
regions 

{'Cont': [<Label | fsaverage, '7Networks_LH_Cont_Cing_1-lh', lh : 1295 vertices>,
  <Label | fsaverage, '7Networks_LH_Cont_Cing_2-lh', lh : 714 vertices>,
  <Label | fsaverage, '7Networks_LH_Cont_OFC_1-lh', lh : 872 vertices>,
  <Label | fsaverage, '7Networks_LH_Cont_PFCl_1-lh', lh : 1089 vertices>,
  <Label | fsaverage, '7Networks_LH_Cont_PFCl_2-lh', lh : 1197 vertices>,
  <Label | fsaverage, '7Networks_LH_Cont_PFCl_3-lh', lh : 1668 vertices>,
  <Label | fsaverage, '7Networks_LH_Cont_PFCl_4-lh', lh : 1660 vertices>,
  <Label | fsaverage, '7Networks_LH_Cont_PFCl_5-lh', lh : 1255 vertices>,
  <Label | fsaverage, '7Networks_LH_Cont_Par_1-lh', lh : 1124 vertices>,
  <Label | fsaverage, '7Networks_LH_Cont_Par_2-lh', lh : 1696 vertices>,
  <Label | fsaverage, '7Networks_LH_Cont_Par_3-lh', lh : 1117 vertices>,
  <Label | fsaverage, '7Networks_LH_Cont_Temp_1-lh', lh : 972 vertices>,
  <Label | fsaverage, '7Networks_LH_Cont_pCun_1-lh', lh : 1226 vertices>,
  <Label | fsaverage, '7Networks_RH_C

In [97]:
src = mne.setup_source_space(
    subject=subject,
    spacing='ico5',
    subjects_dir=subjects_dir,
    add_dist=False
)


Setting up the source space with the following parameters:

SUBJECTS_DIR = C:\Users\hugop\mne_data\MNE-fsaverage-data
Subject      = fsaverage
Surface      = white
Icosahedron subdivision grade 5

>>> 1. Creating the source space...

Doing the icosahedral vertex picking...
Loading C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\lh.white...
Mapping lh fsaverage -> ico (5) ...
    Triangle neighbors and vertex normals...
Loading geometry from C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\lh.sphere...
Setting up the triangulation for the decimated surface...
loaded lh.white 10242/163842 selected to source space (ico = 5)

Loading C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\rh.white...
Mapping rh fsaverage -> ico (5) ...
    Triangle neighbors and vertex normals...
Loading geometry from C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\rh.sphere...
Setting up the triangulation for the decimated surface...
loaded rh.white 10242/163842 selected to 

In [98]:
hemi = 0 # 0 = left | 1 = right


rr = src[hemi]['rr']
vertno = src[hemi]['vertno'] 
coords = rr[vertno] 

In [99]:
y_ant = 0.02    # avant
y_post = -0.04  # arrière
z_sup = 0.02    # haut
z_inf = -0.02   # bas

x = coords[:, 0]
y = coords[:, 1]
z = coords[:, 2]

In [100]:
print(coords.min(axis=0))
print(coords.max(axis=0))

xmin, ymin, zmin = coords.min(axis=0)
xmax, ymax, zmax = coords.max(axis=0)

temporal_mask = (z < z_inf) & (y > y_post) & (y < y_ant)
temporal_vertices = vertno[temporal_mask]

occipital_mask = y < y_post
occipital_vertices = vertno[occipital_mask]

frontal_mask = y > y_ant
frontal_vertices = vertno[frontal_mask]

used = temporal_mask | occipital_mask | frontal_mask
parietal_mask = ~used
parietal_vertices = vertno[parietal_mask]


[-0.0656627  -0.10240875 -0.04242961]
[0.00179359 0.0647402  0.07524361]


In [101]:
atlas = {
    "frontal": frontal_vertices,
    "parietal": parietal_vertices,
    "occipital": occipital_vertices,
    "temporal": temporal_vertices,
    
    # "temporal_ant": temp_ant,
    # "temporal_post": temp_post,
    # "temporal_sup": temp_sup,
    # "temporal_inf": temp_inf,
}

In [102]:
# print(list(regions.keys()))

# index_regions = {}
# for region in regions.keys():
#     index_regions[region+'_lh'] = []
#     index_regions[region+'_rh'] = []

network = list(regions.keys())[2]
regions[network]

[<Label | fsaverage, '7Networks_LH_DorsAttn_FEF_1-lh', lh : 1591 vertices>,
 <Label | fsaverage, '7Networks_LH_DorsAttn_FEF_2-lh', lh : 1167 vertices>,
 <Label | fsaverage, '7Networks_LH_DorsAttn_Post_1-lh', lh : 2063 vertices>,
 <Label | fsaverage, '7Networks_LH_DorsAttn_Post_10-lh', lh : 1653 vertices>,
 <Label | fsaverage, '7Networks_LH_DorsAttn_Post_2-lh', lh : 960 vertices>,
 <Label | fsaverage, '7Networks_LH_DorsAttn_Post_3-lh', lh : 1445 vertices>,
 <Label | fsaverage, '7Networks_LH_DorsAttn_Post_4-lh', lh : 1533 vertices>,
 <Label | fsaverage, '7Networks_LH_DorsAttn_Post_5-lh', lh : 1375 vertices>,
 <Label | fsaverage, '7Networks_LH_DorsAttn_Post_6-lh', lh : 1211 vertices>,
 <Label | fsaverage, '7Networks_LH_DorsAttn_Post_7-lh', lh : 1532 vertices>,
 <Label | fsaverage, '7Networks_LH_DorsAttn_Post_8-lh', lh : 1189 vertices>,
 <Label | fsaverage, '7Networks_LH_DorsAttn_Post_9-lh', lh : 1527 vertices>,
 <Label | fsaverage, '7Networks_LH_DorsAttn_PrCv_1-lh', lh : 1283 vertices>,
 

In [103]:
sfreq = 512
time = np.arange(0,1,1/sfreq)
signal = np.zeros(len(time))
signal[len(time)//2:len(time)//2+len(time)//4] = 1


In [104]:
# roi_lh = next(l for l in regions[network] if l.hemi == 'lh')
vertice_lh = []
vertice_rh = []

for r in regions[network]:
    if r.hemi == 'lh':
        vertice_lh.append(r.vertices)
        
    elif r.hemi == 'rh':
        vertice_rh.append(r.vertices)

vertice_lh = np.concatenate(vertice_lh)
vertice_rh = np.concatenate(vertice_rh)

print(f"avant : len lh = {len(vertice_lh)} | len rh = {len(vertice_rh)}")

vertice_lh = np.unique(vertice_lh)
vertice_rh = np.unique(vertice_rh)

print(f"apres : len lh = {len(vertice_lh)} | len rh = {len(vertice_rh)}")

vertice_lh

avant : len lh = 18529 | len rh = 19549
apres : len lh = 18529 | len rh = 19549


array([     1,     18,     31, ..., 162362, 162363, 163069],
      shape=(18529,))

In [105]:
n_lh = src[0]['nuse']
n_rh = src[1]['nuse']
data = np.zeros((n_lh + n_rh, len(time)))

# ===== LH =====
src_vertices = src[0]['vertno']
inter = np.intersect1d(src_vertices, vertice_lh)
idx = np.searchsorted(src_vertices, inter)
data[idx, :] = signal

# ===== RH =====
src_vertices = src[1]['vertno']
inter = np.intersect1d(src_vertices, vertice_rh)
idx = np.searchsorted(src_vertices, inter)
data[n_lh + idx, :] = signal

print("nonzero in RH:", np.count_nonzero(data[n_lh:, :]))
print("nonzero in LH:", np.count_nonzero(data[:n_lh, :]))

nonzero in RH: 156160
nonzero in LH: 148480


In [106]:
stc = mne.SourceEstimate(
    data,
    vertices=[src[0]['vertno'], src[1]['vertno']],
    tmin=0,
    tstep=0.01,
    subject=subject
)

In [107]:
print("nonzero in RH:", np.count_nonzero(data[n_lh:, :]))
print("nonzero in LH:", np.count_nonzero(data[:n_lh, :]))

nonzero in RH: 156160
nonzero in LH: 148480


In [108]:
import os
os.environ["QT_API"] = "pyqt5"

import mne
mne.viz.set_3d_backend("pyvista")

stc.plot(subjects_dir=subjects_dir,hemi = 'both', clim=dict(kind='value', lims=[0, 0.5, 1]))